## **Parameter for FY & Half Year**

In [0]:
# Databricks notebook (%python)
dbutils.widgets.text("pg_fy", "FY2425")  # 例: FY2425
dbutils.widgets.text("pg_half", "2H") # 例: 2H

## **Zakka (Zakka + Gillette) 完成版**

In [0]:
%sql
WITH
/* -------------------------
   0) パラメータ
   ------------------------- */
params AS (
  SELECT
    -- 'FY2425' AS fy_year_name,   -- ★ここだけ変更しやすく
    -- '2H'     AS fy_half_year_name
    -- 変数方式:
     '${pg_fy}'  AS fy_year_name,
     '${pg_half}' AS fy_half_year_name
),

/* -------------------------
   1) 対象期間 FY / Half Year → 日付 & Quarter
   ------------------------- */
cal_fy_hy AS (
  SELECT
    to_date(c.day_date) AS day_date,
    c.fy_quarter_name   AS pg_quarter
  FROM hive_metastore.japan_gold.calender_dim_vw c
  JOIN params p
    ON c.fy_year_name = p.fy_year_name
   AND c.fy_half_year_name = p.fy_half_year_name
),
cal_bounds AS (
  SELECT
    CAST(MIN(day_date) AS TIMESTAMP)                    AS start_ts,
    CAST((MAX(day_date) + INTERVAL 1 DAY) AS TIMESTAMP) AS end_ts_excl
  FROM cal_fy_hy
),

/* -------------------------
   2) Org（org5_keyごと最新）
   ※ o.* は使わず必要列のみ（system column回避）
   ------------------------- */
org5_latest AS (
  SELECT
    z.org5_key,
    z.org2_name AS org2,
    z.org3_name AS org3,
    z.org4_name AS org4,
    z.org5_name AS org5
  FROM (
    SELECT
      o.org5_key,
      o.org2_name,
      o.org3_name,
      o.org4_name,
      o.org5_name,
      o.validity_end_period_key,
      ROW_NUMBER() OVER (PARTITION BY o.org5_key ORDER BY o.validity_end_period_key DESC) AS rn
    FROM hive_metastore.japan_gold.sl_organization_dim_vw o
    WHERE o.org5_key IS NOT NULL
      AND o.validity_end_period_key IS NOT NULL
  ) z
  WHERE z.rn = 1
),

/* =========================================================
   =========================
   A) Zakka ブロック（①）
   =========================
   ========================================================= */

/* Zakka 7カテゴリ（category_dim側の名称に統一） */
zakka_cat AS (
  SELECT 'Laundry' AS category_name UNION ALL
  SELECT 'Fabric Enhancer' UNION ALL
  SELECT 'Dish Care' UNION ALL
  SELECT 'Air Care' UNION ALL
  SELECT 'Baby Care' UNION ALL
  SELECT 'Feminine Care' UNION ALL
  SELECT 'Hair Care'
),
cat_dim_zakka AS (
  SELECT /*+ BROADCAST */
    cd.category_key,
    cd.category_name
  FROM hive_metastore.japan_gold.category_dim_vw cd
  JOIN zakka_cat z
    ON cd.category_name = z.category_name
),

/* product -> category_key（jp_category_nameで付与） */
zakka_prod_to_cat AS (
  SELECT /*+ BROADCAST */
    p.prod_key,
    cz.category_key,
    cz.category_name
  FROM hive_metastore.japan_gold.product_dim_vw p
  JOIN cat_dim_zakka cz
    ON p.jp_category_name = cz.category_name
),

/* giv_dimension：Zakkaカテゴリのみ & mapping_key×category_keyで end_date 最新 */
zakka_giv_latest AS (
  SELECT
    x.customer_org_mapping_key,
    x.category_key,
    x.org5_key,
    x.end_date,
    x.fn_code,
    x.fn_name
  FROM (
    SELECT
      gd.customer_org_mapping_key,
      gd.Category_key AS category_key,
      gd.org5_key,
      gd.end_date,
      gd.FN_Code AS fn_code,
      gd.FN_Name AS fn_name,
      ROW_NUMBER() OVER (
        PARTITION BY gd.customer_org_mapping_key, gd.Category_key
        ORDER BY gd.end_date DESC
      ) AS rn
    FROM hive_metastore.japan_gold.giv_dimension_dim_vw gd
    JOIN cat_dim_zakka cz
      ON gd.Category_key = cz.category_key
  ) x
  WHERE x.rn = 1
),

/* FN Name（最新版）: fn_code単位で end_date 最大の行から取得 */
zakka_fn_latest_name AS (
  SELECT
    t.fn_code,
    t.fn_name_latest
  FROM (
    SELECT
      gd.FN_Code AS fn_code,
      gd.FN_Name AS fn_name_latest,
      ROW_NUMBER() OVER (
        PARTITION BY gd.FN_Code
        ORDER BY gd.end_date DESC
      ) AS rn
    FROM hive_metastore.japan_gold.giv_dimension_dim_vw gd
    WHERE gd.FN_Code IS NOT NULL
      AND gd.FN_Name IS NOT NULL
      AND gd.end_date IS NOT NULL
  ) t
  WHERE t.rn = 1
),

/* Factを mapping_key×quarter で集約（高速） */
zakka_ship_qtr AS (
  SELECT
    sh.customer_org_mapping_key,
    cal.pg_quarter,
    SUM(sh.gross_transact_amt) AS ship_giv
  FROM hive_metastore.japan_gold.shipments_fct_vw sh
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(sh.day_date) = cal.day_date
  WHERE sh.day_date >= b.start_ts AND sh.day_date < b.end_ts_excl
  GROUP BY sh.customer_org_mapping_key, cal.pg_quarter
),
zakka_move_qtr AS (
  SELECT
    mv.customer_org_mapping_key,
    cal.pg_quarter,
    SUM(mv.gross_transact_amt) AS move_giv,
    SUM(mv.net_transact_amt)   AS move_niv
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw mv
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(mv.day_date) = cal.day_date
  WHERE mv.day_date >= b.start_ts AND mv.day_date < b.end_ts_excl
  GROUP BY mv.customer_org_mapping_key, cal.pg_quarter
),
zakka_garp_qtr AS (
  SELECT
    g.customer_org_mapping_key,
    cal.pg_quarter,
    SUM(g.garp_value) AS garp
  FROM hive_metastore.japan_gold.giv_garp_dim_vw g
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(g.day_date) = cal.day_date
  WHERE g.day_date >= b.start_ts AND g.day_date < b.end_ts_excl
  GROUP BY g.customer_org_mapping_key, cal.pg_quarter
),
zakka_outbound_qtr AS (
  SELECT
    o.customer_org_mapping_key,
    cal.pg_quarter,
    SUM(o.giv_outbound) AS outbound
  FROM hive_metastore.japan_gold.giv_outbound_vw o
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(o.day_date) = cal.day_date
  WHERE o.day_date >= b.start_ts AND o.day_date < b.end_ts_excl
  GROUP BY o.customer_org_mapping_key, cal.pg_quarter
),

zakka_all_keys AS (
  SELECT customer_org_mapping_key, pg_quarter FROM zakka_ship_qtr
  UNION
  SELECT customer_org_mapping_key, pg_quarter FROM zakka_move_qtr
  UNION
  SELECT customer_org_mapping_key, pg_quarter FROM zakka_garp_qtr
  UNION
  SELECT customer_org_mapping_key, pg_quarter FROM zakka_outbound_qtr
),

zakka_wide_qtr AS (
  SELECT
    k.customer_org_mapping_key,
    k.pg_quarter,
    COALESCE(s.ship_giv, 0.0) AS ship_giv,
    COALESCE(m.move_giv, 0.0) AS move_giv,
    COALESCE(m.move_niv, 0.0) AS move_niv,
    COALESCE(g.garp, 0.0)     AS garp,
    COALESCE(o.outbound, 0.0) AS outbound
  FROM zakka_all_keys k
  LEFT JOIN zakka_ship_qtr     s ON k.customer_org_mapping_key=s.customer_org_mapping_key AND k.pg_quarter=s.pg_quarter
  LEFT JOIN zakka_move_qtr     m ON k.customer_org_mapping_key=m.customer_org_mapping_key AND k.pg_quarter=m.pg_quarter
  LEFT JOIN zakka_garp_qtr     g ON k.customer_org_mapping_key=g.customer_org_mapping_key AND k.pg_quarter=g.pg_quarter
  LEFT JOIN zakka_outbound_qtr o ON k.customer_org_mapping_key=o.customer_org_mapping_key AND k.pg_quarter=o.pg_quarter
),

/* mapping_key×category_key に category_name を付与（Zakkaのみ） */
zakka_mapping_enriched AS (
  SELECT
    gl.customer_org_mapping_key,
    gl.category_key,
    cz.category_name AS category,
    gl.org5_key,
    gl.end_date,
    gl.fn_code
  FROM zakka_giv_latest gl
  JOIN cat_dim_zakka cz
    ON gl.category_key = cz.category_key
  WHERE gl.fn_code IS NOT NULL
),

/* Orgは fn_code×category_key で end_date最新 */
zakka_org_fn_cat AS (
  SELECT fn_code, category_key, org5_key
  FROM (
    SELECT
      me.fn_code,
      me.category_key,
      me.org5_key,
      ROW_NUMBER() OVER (
        PARTITION BY me.fn_code, me.category_key
        ORDER BY me.end_date DESC
      ) AS rn
    FROM zakka_mapping_enriched me
    WHERE me.org5_key IS NOT NULL
  ) x
  WHERE x.rn = 1
),

/* baseline KPI */
zakka_kpi_base AS (
  SELECT
    me.fn_code,
    me.category,
    'Zakka' AS z_e,
    wq.pg_quarter AS quarter,
    me.category_key,
    SUM(wq.ship_giv + wq.move_giv) AS ships_in_giv,
    SUM(wq.ship_giv)              AS ship_giv,
    SUM(wq.move_giv)              AS move_giv,
    SUM(wq.move_niv)              AS move_niv,
    SUM(wq.garp)                  AS garp,
    SUM(wq.outbound)              AS outbound
  FROM zakka_wide_qtr wq
  JOIN zakka_mapping_enriched me
    ON wq.customer_org_mapping_key = me.customer_org_mapping_key
  GROUP BY me.fn_code, me.category, wq.pg_quarter, me.category_key
),

/* ★自動パッチ（Z-DC & FN% の baseline欠損を補完） */
zakka_rt_fn_dc AS (
  SELECT DISTINCT
    c.jp_local_fn_code_2 AS fn_code,
    c.jp_local_fn_name_2 AS fn_name
  FROM hive_metastore.japan_gold.customer_japan_dim_vw c
  WHERE c.jp_cust_parent_group_name='RT'
    AND trim(c.jp_local_zakka_dc_nc_2)='Z-DC'
    AND c.jp_local_fn_code_2 IS NOT NULL
    AND c.jp_local_fn_name_2 LIKE 'FN%'
),
zakka_base_fn AS (
  SELECT DISTINCT fn_code FROM zakka_kpi_base
),
zakka_missing_dc_fn AS (
  SELECT r.fn_code, r.fn_name
  FROM zakka_rt_fn_dc r
  LEFT JOIN zakka_base_fn b
    ON r.fn_code = b.fn_code
  WHERE b.fn_code IS NULL
),

zakka_dc_ship_patch AS (
  SELECT
    c.jp_local_fn_code_2 AS fn_code,
    ptc.category_name AS category,
    'Zakka' AS z_e,
    cal.pg_quarter AS quarter,
    ptc.category_key,
    SUM(sh.gross_transact_amt) AS ships_in_giv,
    SUM(sh.gross_transact_amt) AS ship_giv,
    CAST(0.0 AS DOUBLE) AS move_giv,
    CAST(0.0 AS DOUBLE) AS move_niv,
    CAST(0.0 AS DOUBLE) AS garp,
    CAST(0.0 AS DOUBLE) AS outbound
  FROM hive_metastore.japan_gold.shipments_fct_vw sh
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal ON to_date(sh.day_date)=cal.day_date
  JOIN hive_metastore.japan_gold.customer_japan_dim_vw c
    ON CAST(sh.whkey AS BIGINT) = CAST(c.cust_key AS BIGINT)
  JOIN zakka_missing_dc_fn md
    ON c.jp_local_fn_code_2 = md.fn_code
  JOIN zakka_prod_to_cat ptc
    ON sh.prod_key = ptc.prod_key
  WHERE sh.day_date >= b.start_ts AND sh.day_date < b.end_ts_excl
  GROUP BY c.jp_local_fn_code_2, ptc.category_name, ptc.category_key, cal.pg_quarter
),
zakka_dc_move_patch AS (
  SELECT
    c.jp_local_fn_code_2 AS fn_code,
    ptc.category_name AS category,
    'Zakka' AS z_e,
    cal.pg_quarter AS quarter,
    ptc.category_key,
    SUM(mv.gross_transact_amt) AS ships_in_giv,
    CAST(0.0 AS DOUBLE) AS ship_giv,
    SUM(mv.gross_transact_amt) AS move_giv,
    SUM(mv.net_transact_amt)   AS move_niv,
    CAST(0.0 AS DOUBLE) AS garp,
    CAST(0.0 AS DOUBLE) AS outbound
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw mv
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal ON to_date(mv.day_date)=cal.day_date
  JOIN hive_metastore.japan_gold.customer_japan_dim_vw c
    ON CAST(mv.cust_key AS BIGINT) = CAST(c.cust_key AS BIGINT)
  JOIN zakka_missing_dc_fn md
    ON c.jp_local_fn_code_2 = md.fn_code
  JOIN zakka_prod_to_cat ptc
    ON mv.prod_key = ptc.prod_key
  WHERE mv.day_date >= b.start_ts AND mv.day_date < b.end_ts_excl
  GROUP BY c.jp_local_fn_code_2, ptc.category_name, ptc.category_key, cal.pg_quarter
),
zakka_kpi_patch AS (
  SELECT
    fn_code, category, z_e, quarter, category_key,
    SUM(ships_in_giv) AS ships_in_giv,
    SUM(ship_giv)     AS ship_giv,
    SUM(move_giv)     AS move_giv,
    SUM(move_niv)     AS move_niv,
    SUM(garp)         AS garp,
    SUM(outbound)     AS outbound
  FROM (
    SELECT * FROM zakka_dc_ship_patch
    UNION ALL
    SELECT * FROM zakka_dc_move_patch
  ) x
  GROUP BY fn_code, category, z_e, quarter, category_key
),
zakka_kpi_final AS (
  SELECT
    fn_code,
    category,
    z_e,
    quarter,
    category_key,
    SUM(ships_in_giv) AS ships_in_giv,
    SUM(ship_giv)     AS ship_giv,
    SUM(move_giv)     AS move_giv,
    SUM(move_niv)     AS move_niv,
    SUM(garp)         AS garp,
    SUM(outbound)     AS outbound
  FROM (
    SELECT * FROM zakka_kpi_base
    UNION ALL
    SELECT * FROM zakka_kpi_patch
  ) allz
  GROUP BY fn_code, category, z_e, quarter, category_key
),

zakka_result AS (
  SELECT
    k.fn_code,
    COALESCE(fnl.fn_name_latest, rt.fn_name, k.fn_code) AS fn_name,
    CASE
      WHEN COALESCE(fnl.fn_name_latest, rt.fn_name) LIKE 'FN%'  THEN 'DC'
      WHEN COALESCE(fnl.fn_name_latest, rt.fn_name) LIKE 'WS#%' THEN 'WS'
      ELSE NULL
    END AS dc_ws,
    k.category,
    k.z_e,
    k.quarter,
    o.org2,
    o.org3,
    o.org4,
    o.org5,
    k.ships_in_giv,
    k.ship_giv,
    k.move_giv,
    k.move_niv,
    k.garp,
    k.outbound,
    1 AS sort_block -- ★UNION後の整列用（不要なら削除）
  FROM zakka_kpi_final k
  LEFT JOIN zakka_fn_latest_name fnl
    ON k.fn_code = fnl.fn_code
  LEFT JOIN zakka_rt_fn_dc rt
    ON k.fn_code = rt.fn_code
  LEFT JOIN zakka_org_fn_cat ofc
    ON k.fn_code=ofc.fn_code AND k.category_key=ofc.category_key
  LEFT JOIN org5_latest o
    ON ofc.org5_key=o.org5_key
),

/* =========================================================
   =========================
   B) Gillette ブロック（②）
   =========================
   ========================================================= */

/* 対象カテゴリ */
gillette_target_cat AS (
  SELECT 'Shave Care' AS category_name UNION ALL
  SELECT 'Oral Care'  UNION ALL
  SELECT 'Appliances'
),
gillette_cat_dim_target AS (
  SELECT /*+ BROADCAST */
    cd.category_key,
    cd.category_name
  FROM hive_metastore.japan_gold.category_dim_vw cd
  JOIN gillette_target_cat t
    ON cd.category_name = t.category_name
),

/* 商品→カテゴリ（jp_category_nameで付与、overlap_76も保持） */
gillette_prod_to_cat AS (
  SELECT /*+ BROADCAST */
    p.prod_key,
    cd.category_key,
    cd.category_name,
    p.jp_pg_item_status_code AS overlap_76
  FROM hive_metastore.japan_gold.product_dim_vw p
  JOIN gillette_cat_dim_target cd
    ON p.jp_category_name = cd.category_name
),

/* Customer / Warehouse（必要列のみ） */
gillette_cust_dim AS (
  SELECT /*+ BROADCAST */
    CAST(c.cust_key AS BIGINT)      AS cust_key,
    c.jp_cust_parent_group_name     AS cust_parent_group,
    c.jp_local_fn_code_2            AS fn_code2,
    c.jp_local_fn_name_2            AS fn_name2,
    c.jp_local_gillette_dc_nc_2     AS gillette_flag2
  FROM hive_metastore.japan_gold.customer_japan_dim_vw c
),
gillette_wh_dim AS (
  SELECT /*+ BROADCAST */
    CAST(w.cust_key AS BIGINT)      AS whkey,
    w.jp_cust_parent_group_name     AS wh_parent_group,
    w.jp_local_fn_code_2            AS wh_fn_code2,
    w.jp_local_fn_name_2            AS wh_fn_name2,
    w.jp_local_gillette_dc_nc_2     AS wh_gillette_flag2
  FROM hive_metastore.japan_gold.warehouse_dim_vw w
),

/* Org（giv_dimension: mapping_key×category_key で end_date 最新） */
gillette_giv_org_latest AS (
  SELECT
    x.customer_org_mapping_key,
    x.category_key,
    x.org5_key,
    x.end_date
  FROM (
    SELECT
      gd.customer_org_mapping_key,
      gd.Category_key AS category_key,
      gd.org5_key,
      gd.end_date,
      ROW_NUMBER() OVER (
        PARTITION BY gd.customer_org_mapping_key, gd.Category_key
        ORDER BY gd.end_date DESC
      ) AS rn
    FROM hive_metastore.japan_gold.giv_dimension_dim_vw gd
  ) x
  WHERE x.rn=1
),

/* GARP / Outbound（mapping_key×category_key×quarter） */
gillette_garp_qtr_cat AS (
  SELECT
    g.customer_org_mapping_key,
    g.sl_category_key AS category_key,
    cal.pg_quarter    AS quarter,
    SUM(g.garp_value) AS garp
  FROM hive_metastore.japan_gold.giv_garp_dim_vw g
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(g.day_date)=cal.day_date
  WHERE g.day_date >= b.start_ts AND g.day_date < b.end_ts_excl
  GROUP BY g.customer_org_mapping_key, g.sl_category_key, cal.pg_quarter
),
gillette_outbound_qtr_cat AS (
  SELECT
    o.customer_org_mapping_key,
    o.sl_category_key AS category_key,
    cal.pg_quarter    AS quarter,
    SUM(o.giv_outbound) AS outbound
  FROM hive_metastore.japan_gold.giv_outbound_vw o
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(o.day_date)=cal.day_date
  WHERE o.day_date >= b.start_ts AND o.day_date < b.end_ts_excl
  GROUP BY o.customer_org_mapping_key, o.sl_category_key, cal.pg_quarter
),

/* --- (b-1) Shave DC: ship+move --- */
gillette_shave_dc_ship AS (
  SELECT
    sh.customer_org_mapping_key,
    sh.customer_org_mapping_key AS customer_org_mapping_key_dup, -- JOINキー保持（可読性）
    ptc.category_key,
    ptc.category_name AS category,
    cal.pg_quarter    AS quarter,
    cd.fn_code2       AS fn_code,
    MAX(cd.fn_name2)  AS fn_name,
    'DC'              AS dc_ws,
    'Zakka'           AS z_e,
    SUM(sh.gross_transact_amt) AS ship_giv,
    CAST(0.0 AS DOUBLE)        AS move_giv,
    CAST(0.0 AS DOUBLE)        AS move_niv
  FROM hive_metastore.japan_gold.shipments_fct_vw sh
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(sh.day_date)=cal.day_date
  JOIN gillette_prod_to_cat ptc
    ON sh.prod_key = ptc.prod_key
   AND ptc.category_name = 'Shave Care'
  JOIN gillette_cust_dim cd
    ON CAST(sh.whkey AS BIGINT) = cd.cust_key
  WHERE sh.day_date >= b.start_ts AND sh.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.gillette_flag2 = 'G-DC'
  GROUP BY sh.customer_org_mapping_key, ptc.category_key, ptc.category_name, cal.pg_quarter, cd.fn_code2
),
gillette_shave_dc_move AS (
  SELECT
    mv.customer_org_mapping_key,
    mv.customer_org_mapping_key AS customer_org_mapping_key_dup,
    ptc.category_key,
    ptc.category_name AS category,
    cal.pg_quarter    AS quarter,
    cd.fn_code2       AS fn_code,
    MAX(cd.fn_name2)  AS fn_name,
    'DC'              AS dc_ws,
    'Zakka'           AS z_e,
    CAST(0.0 AS DOUBLE)        AS ship_giv,
    SUM(mv.gross_transact_amt) AS move_giv,
    SUM(mv.net_transact_amt)   AS move_niv
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw mv
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(mv.day_date)=cal.day_date
  JOIN gillette_prod_to_cat ptc
    ON mv.prod_key = ptc.prod_key
   AND ptc.category_name = 'Shave Care'
  JOIN gillette_cust_dim cd
    ON CAST(mv.cust_key AS BIGINT) = cd.cust_key
  WHERE mv.day_date >= b.start_ts AND mv.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.gillette_flag2 = 'G-DC'
  GROUP BY mv.customer_org_mapping_key, ptc.category_key, ptc.category_name, cal.pg_quarter, cd.fn_code2
),

/* --- (C-1) Oral/AP DC Move: moveのみ + WH parent != WS Electro --- */
gillette_oral_ap_dc_move AS (
  SELECT
    mv.customer_org_mapping_key,
    mv.customer_org_mapping_key AS customer_org_mapping_key_dup,
    ptc.category_key,
    ptc.category_name AS category,
    cal.pg_quarter    AS quarter,
    cd.fn_code2       AS fn_code,
    MAX(cd.fn_name2)  AS fn_name,
    'DC'              AS dc_ws,
    'Zakka'           AS z_e,
    CAST(0.0 AS DOUBLE)        AS ship_giv,
    SUM(mv.gross_transact_amt) AS move_giv,
    SUM(mv.net_transact_amt)   AS move_niv
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw mv
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(mv.day_date)=cal.day_date
  JOIN gillette_prod_to_cat ptc
    ON mv.prod_key = ptc.prod_key
   AND ptc.category_name IN ('Oral Care','Appliances')
  JOIN gillette_cust_dim cd
    ON CAST(mv.cust_key AS BIGINT) = cd.cust_key
  JOIN gillette_wh_dim wd
    ON CAST(mv.whkey AS BIGINT) = wd.whkey
  WHERE mv.day_date >= b.start_ts AND mv.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.gillette_flag2 = 'G-DC'
    AND wd.wh_parent_group <> 'WS Electro'
  GROUP BY mv.customer_org_mapping_key, ptc.category_key, ptc.category_name, cal.pg_quarter, cd.fn_code2
),

/* --- (C-2) Oral/AP DC Ship: shipのみ + overlap=Zakka-* --- */
gillette_oral_ap_dc_ship AS (
  SELECT
    sh.customer_org_mapping_key,
    sh.customer_org_mapping_key AS customer_org_mapping_key_dup,
    ptc.category_key,
    ptc.category_name AS category,
    cal.pg_quarter    AS quarter,
    cd.fn_code2       AS fn_code,
    MAX(cd.fn_name2)  AS fn_name,
    'DC'              AS dc_ws,
    'Zakka'           AS z_e,
    SUM(sh.gross_transact_amt) AS ship_giv,
    CAST(0.0 AS DOUBLE)        AS move_giv,
    CAST(0.0 AS DOUBLE)        AS move_niv
  FROM hive_metastore.japan_gold.shipments_fct_vw sh
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(sh.day_date)=cal.day_date
  JOIN gillette_prod_to_cat ptc
    ON sh.prod_key = ptc.prod_key
   AND ptc.category_name IN ('Oral Care','Appliances')
  JOIN gillette_cust_dim cd
    ON CAST(sh.whkey AS BIGINT) = cd.cust_key
  WHERE sh.day_date >= b.start_ts AND sh.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.gillette_flag2 = 'G-DC'
    AND ptc.overlap_76 IN ('Zakka-Oral','Zakka-Appliances')
  GROUP BY sh.customer_org_mapping_key, ptc.category_key, ptc.category_name, cal.pg_quarter, cd.fn_code2
),

/* --- (C-3) Gillette NC: moveのみ（warehouse FNで集計） --- */
gillette_nc_move AS (
  SELECT
    mv.customer_org_mapping_key,
    mv.customer_org_mapping_key AS customer_org_mapping_key_dup,
    ptc.category_key,
    ptc.category_name AS category,
    cal.pg_quarter    AS quarter,
    wd.wh_fn_code2    AS fn_code,
    MAX(wd.wh_fn_name2) AS fn_name,
    'WS'              AS dc_ws,
    'Zakka'           AS z_e,
    CAST(0.0 AS DOUBLE)        AS ship_giv,
    SUM(mv.gross_transact_amt) AS move_giv,
    SUM(mv.net_transact_amt)   AS move_niv
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw mv
  CROSS JOIN cal_bounds b
  JOIN cal_fy_hy cal
    ON to_date(mv.day_date)=cal.day_date
  JOIN gillette_prod_to_cat ptc
    ON mv.prod_key = ptc.prod_key
   AND ptc.category_name IN ('Shave Care','Oral Care','Appliances')
  JOIN gillette_cust_dim cd
    ON CAST(mv.cust_key AS BIGINT) = cd.cust_key
  JOIN gillette_wh_dim wd
    ON CAST(mv.whkey AS BIGINT) = wd.whkey
  WHERE mv.day_date >= b.start_ts AND mv.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.gillette_flag2 = 'G-NC'
    AND wd.wh_gillette_flag2 = 'G-DC'
    AND wd.wh_parent_group <> 'WS Electro'
  GROUP BY mv.customer_org_mapping_key, ptc.category_key, ptc.category_name, cal.pg_quarter, wd.wh_fn_code2
),

/* シナリオ統合 */
gillette_scenario_rows AS (
  SELECT * FROM gillette_shave_dc_ship
  UNION ALL SELECT * FROM gillette_shave_dc_move
  UNION ALL SELECT * FROM gillette_oral_ap_dc_move
  UNION ALL SELECT * FROM gillette_oral_ap_dc_ship
  UNION ALL SELECT * FROM gillette_nc_move
),

gillette_scenario_with_go AS (
  SELECT
    sr.fn_code,
    MAX(sr.fn_name) AS fn_name,
    sr.dc_ws,
    sr.category,
    sr.z_e,
    sr.quarter,
    sr.category_key,
    SUM(sr.ship_giv) AS ship_giv,
    SUM(sr.move_giv) AS move_giv,
    SUM(sr.move_niv) AS move_niv,
    SUM(sr.ship_giv + sr.move_giv) AS ships_in_giv,
    SUM(COALESCE(gq.garp,0.0)) AS garp,
    SUM(COALESCE(oq.outbound,0.0)) AS outbound
  FROM gillette_scenario_rows sr
  LEFT JOIN gillette_garp_qtr_cat gq
    ON sr.customer_org_mapping_key = gq.customer_org_mapping_key
   AND sr.category_key = gq.category_key
   AND sr.quarter = gq.quarter
  LEFT JOIN gillette_outbound_qtr_cat oq
    ON sr.customer_org_mapping_key = oq.customer_org_mapping_key
   AND sr.category_key = oq.category_key
   AND sr.quarter = oq.quarter
  GROUP BY sr.fn_code, sr.dc_ws, sr.category, sr.z_e, sr.quarter, sr.category_key
),

/* Org付与（fn_code×category_key の end_date最新） */
gillette_org_candidates AS (
  SELECT DISTINCT
    sr.fn_code,
    sr.category_key,
    gol.org5_key,
    gol.end_date
  FROM gillette_scenario_rows sr
  LEFT JOIN gillette_giv_org_latest gol
    ON sr.customer_org_mapping_key = gol.customer_org_mapping_key
   AND sr.category_key = gol.category_key
),
gillette_org_fn_cat AS (
  SELECT
    x.fn_code,
    x.category_key,
    x.org5_key
  FROM (
    SELECT
      oc.fn_code,
      oc.category_key,
      oc.org5_key,
      ROW_NUMBER() OVER (
        PARTITION BY oc.fn_code, oc.category_key
        ORDER BY oc.end_date DESC NULLS LAST
      ) AS rn
    FROM gillette_org_candidates oc
    WHERE oc.org5_key IS NOT NULL
  ) x
  WHERE x.rn = 1
),

gillette_result AS (
  SELECT
    s.fn_code,
    s.fn_name,
    s.dc_ws,
    s.category,
    s.z_e,
    s.quarter,
    o.org2,
    o.org3,
    o.org4,
    o.org5,
    s.ships_in_giv,
    s.ship_giv,
    s.move_giv,
    s.move_niv,
    s.garp,
    s.outbound,
    2 AS sort_block -- ★UNION後の整列用（不要なら削除）
  FROM gillette_scenario_with_go s
  LEFT JOIN gillette_org_fn_cat ofc
    ON s.fn_code=ofc.fn_code AND s.category_key=ofc.category_key
  LEFT JOIN org5_latest o
    ON ofc.org5_key=o.org5_key
)

/* =========================================================
   Final: Zakka + Gillette を1本化
   ========================================================= */
SELECT
  r.fn_code,
  r.fn_name,
  r.dc_ws,
  r.category,
  r.z_e,
  r.quarter,
  r.org2,
  r.org3,
  r.org4,
  r.org5,
  r.ships_in_giv,
  r.ship_giv,
  r.move_giv,
  r.move_niv,
  r.garp,
  r.outbound
FROM (
  SELECT * FROM zakka_result
  UNION ALL
  SELECT * FROM gillette_result
) r
ORDER BY
  r.fn_code, r.quarter, r.category, r.dc_ws
;

## **Electro 完成版**

In [0]:
%sql
WITH
/* =========================================================
   0) 期間パラメータ（変更はここだけ）
   ========================================================= */
params AS (
  SELECT
    -- 'FY2425' AS fy_year_name,   -- ★ここだけ変更しやすく
    -- '2H'     AS fy_half_year_name
    -- 変数方式:
     '${pg_fy}'  AS fy_year_name,
     '${pg_half}' AS fy_half_year_name
),

/* FY×HalfYear の日付と Quarter を確定（445のPG Quarter名を使用） */
cal_fy_half AS (
  SELECT
    to_date(c.day_date)  AS day_date,
    c.fy_quarter_name    AS pg_quarter
  FROM hive_metastore.japan_gold.calender_dim_vw c
  JOIN params p
    ON c.fy_year_name = p.fy_year_name
   AND c.fy_half_year_name = p.fy_half_year_name
),
cal_bounds AS (
  SELECT
    CAST(MIN(day_date) AS TIMESTAMP)                    AS start_ts,
    CAST((MAX(day_date) + INTERVAL 1 DAY) AS TIMESTAMP) AS end_ts_excl
  FROM cal_fy_half
),

/* =========================================================
   1) 対象カテゴリ（Oral Care / Appliances）
   ========================================================= */
target_cat AS (
  SELECT 'Oral Care' AS category_name UNION ALL
  SELECT 'Appliances'
),
cat_dim_target AS (
  SELECT /*+ BROADCAST */
    cd.category_key,
    cd.category_name
  FROM hive_metastore.japan_gold.category_dim_vw cd
  JOIN target_cat tc
    ON cd.category_name = tc.category_name
),

/* =========================================================
   2) 商品→カテゴリ付与 + overlap判定（76.Overlapping SKU (Electro)）
   - product_dim_vw.jp_pg_item_status_code を overlap_76 として利用
   ========================================================= */
prod_to_cat AS (
  SELECT /*+ BROADCAST */
    p.prod_key,
    cdt.category_key,
    cdt.category_name AS category_name,
    p.jp_pg_item_status_code AS overlap_76
  FROM hive_metastore.japan_gold.product_dim_vw p
  JOIN cat_dim_target cdt
    ON p.jp_category_name = cdt.category_name
),

/* =========================================================
   3) Customer / Warehouse / Site（必要列のみ）
   - Direct Ship対策：ship_to_site_key→site_dim→own_party_key→customer
   ========================================================= */
cust_dim AS (
  SELECT /*+ BROADCAST */
    CAST(c.cust_key AS BIGINT)       AS cust_key,
    c.jp_cust_parent_group_name      AS cust_parent_group,
    c.jp_local_fn_code_2             AS fn_code2,
    c.jp_local_fn_name_2             AS fn_name2,
    c.jp_local_electro_direct_call_2 AS electro_flag2
  FROM hive_metastore.japan_gold.customer_japan_dim_vw c
),
wh_dim AS (
  SELECT /*+ BROADCAST */
    CAST(w.cust_key AS BIGINT)       AS whkey,
    w.jp_cust_parent_group_name      AS wh_parent_group,
    w.jp_local_fn_code_2             AS wh_fn_code2,
    w.jp_local_fn_name_2             AS wh_fn_name2
  FROM hive_metastore.japan_gold.warehouse_dim_vw w
),
site_dim AS (
  SELECT /*+ BROADCAST */
    s.site_key,
    CAST(s.own_party_key AS BIGINT) AS own_party_key
  FROM hive_metastore.japan_gold.site_dim_vw s
),

/* =========================================================
   4) Org付与用：giv_dimension end_date 最新（mapping_key×category）
      さらに FN×Category で end_date 最新を最終採用する
   ========================================================= */
giv_org_latest_by_mapping AS (
  SELECT
    y.customer_org_mapping_key,
    y.category_key,
    y.org5_key,
    y.end_date
  FROM (
    SELECT
      gd.customer_org_mapping_key,
      gd.Category_key AS category_key,
      gd.org5_key,
      gd.end_date,
      ROW_NUMBER() OVER (
        PARTITION BY gd.customer_org_mapping_key, gd.Category_key
        ORDER BY gd.end_date DESC
      ) AS rn
    FROM hive_metastore.japan_gold.giv_dimension_dim_vw gd
  ) y
  WHERE y.rn = 1
),
org5_latest AS (
  SELECT
    x.org5_key,
    x.org2_name AS org2,
    x.org3_name AS org3,
    x.org4_name AS org4,
    x.org5_name AS org5
  FROM (
    SELECT
      o.org5_key,
      o.org2_name,
      o.org3_name,
      o.org4_name,
      o.org5_name,
      ROW_NUMBER() OVER (
        PARTITION BY o.org5_key
        ORDER BY o.validity_end_period_key DESC
      ) AS rn
    FROM hive_metastore.japan_gold.sl_organization_dim_vw o
    WHERE o.org5_key IS NOT NULL
      AND o.validity_end_period_key IS NOT NULL
  ) x
  WHERE x.rn = 1
),

/* =========================================================
   5) GARP / Outbound（mapping_key×category_key×quarter）
   ========================================================= */
garp_qtr_cat AS (
  SELECT
    g.customer_org_mapping_key,
    g.sl_category_key AS category_key,
    cal.pg_quarter    AS quarter,
    SUM(g.garp_value) AS garp
  FROM hive_metastore.japan_gold.giv_garp_dim_vw g
  CROSS JOIN cal_bounds b
  JOIN cal_fy_half cal
    ON to_date(g.day_date) = cal.day_date
  WHERE g.day_date >= b.start_ts AND g.day_date < b.end_ts_excl
  GROUP BY g.customer_org_mapping_key, g.sl_category_key, cal.pg_quarter
),
outbound_qtr_cat AS (
  SELECT
    o.customer_org_mapping_key,
    o.sl_category_key AS category_key,
    cal.pg_quarter    AS quarter,
    SUM(o.giv_outbound) AS outbound
  FROM hive_metastore.japan_gold.giv_outbound_vw o
  CROSS JOIN cal_bounds b
  JOIN cal_fy_half cal
    ON to_date(o.day_date) = cal.day_date
  WHERE o.day_date >= b.start_ts AND o.day_date < b.end_ts_excl
  GROUP BY o.customer_org_mapping_key, o.sl_category_key, cal.pg_quarter
),

/* =========================================================
   6) D-1 Electro DC Move
      - Category: Oral Care / Appliances
      - customer Electro Flag2 = 'E-DC'
      - warehouse Parent Group = 'WS Electro'
      - overlap NOT IN ('Zakka-Oral','Zakka-Appliances')
      - 集計：Move(gross/net)のみ、キー：customer FN Code2
   ========================================================= */
d1_electro_dc_move AS (
  SELECT
    mv.customer_org_mapping_key,
    ptc.category_key,
    ptc.category_name        AS category,
    cal.pg_quarter           AS quarter,

    cd.fn_code2              AS fn_code,
    MAX(cd.fn_name2)         AS fn_name,      -- 注意：Keyにしない（要件通りMAXで代表値）
    'DC'                     AS dc_ws,
    'Electro'                AS z_e,

    CAST(0.0 AS DOUBLE)      AS ship_giv,
    SUM(mv.gross_transact_amt) AS move_giv,   -- 指示: indirect_niv_value_gross
    SUM(mv.net_transact_amt)   AS move_niv    -- 指示: indirect_niv_value_net
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw mv
  CROSS JOIN cal_bounds b
  JOIN cal_fy_half cal
    ON to_date(mv.day_date) = cal.day_date
  JOIN prod_to_cat ptc
    ON mv.prod_key = ptc.prod_key
  JOIN site_dim sd
    ON mv.ship_to_site_key = sd.site_key         -- customer(店舗)特定のため
  JOIN cust_dim cd
    ON sd.own_party_key = cd.cust_key            -- ★RTかつE-DC判定はcustomer側
  JOIN wh_dim wd
    ON CAST(mv.whkey AS BIGINT) = wd.whkey       -- ★WH条件はwarehouse側
  WHERE mv.day_date >= b.start_ts AND mv.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.electro_flag2 = 'E-DC'
    AND wd.wh_parent_group = 'WS Electro'
    AND (ptc.overlap_76 IS NULL OR ptc.overlap_76 NOT IN ('Zakka-Oral','Zakka-Appliances'))
    AND cd.fn_code2 IS NOT NULL
  GROUP BY
    mv.customer_org_mapping_key,
    ptc.category_key,
    ptc.category_name,
    cal.pg_quarter,
    cd.fn_code2
),

/* =========================================================
   7) D-2 Electro DC Ship
      - Category: Oral Care / Appliances
      - customer Electro Flag2 = 'E-DC'
      - overlap NOT IN ('Zakka-Oral','Zakka-Appliances')
      - 集計：Ship(gross)のみ、キー：customer FN Code2
      - Direct Shipを取りこぼさないため ship_to_site_key→site→customer で紐付け
   ========================================================= */
d2_electro_dc_ship AS (
  SELECT
    sh.customer_org_mapping_key,
    ptc.category_key,
    ptc.category_name          AS category,
    cal.pg_quarter             AS quarter,

    cd.fn_code2                AS fn_code,
    MAX(cd.fn_name2)           AS fn_name,
    'DC'                       AS dc_ws,
    'Electro'                  AS z_e,

    SUM(sh.gross_transact_amt) AS ship_giv,    -- shipment_giv_value_gross
    CAST(0.0 AS DOUBLE)        AS move_giv,
    CAST(0.0 AS DOUBLE)        AS move_niv
  FROM hive_metastore.japan_gold.shipments_fct_vw sh
  CROSS JOIN cal_bounds b
  JOIN cal_fy_half cal
    ON to_date(sh.day_date) = cal.day_date
  JOIN prod_to_cat ptc
    ON sh.prod_key = ptc.prod_key
  JOIN site_dim sd
    ON sh.ship_to_site_key = sd.site_key       -- ★Direct/Indirect問わず店舗起点でcustomerへ
  JOIN cust_dim cd
    ON sd.own_party_key = cd.cust_key
  WHERE sh.day_date >= b.start_ts AND sh.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.electro_flag2 = 'E-DC'
    AND (ptc.overlap_76 IS NULL OR ptc.overlap_76 NOT IN ('Zakka-Oral','Zakka-Appliances'))
    AND cd.fn_code2 IS NOT NULL
  GROUP BY
    sh.customer_org_mapping_key,
    ptc.category_key,
    ptc.category_name,
    cal.pg_quarter,
    cd.fn_code2
),

/* =========================================================
   8) D-3 Electro NC
      - Category: Oral Care / Appliances
      - customer Electro Flag2 = 'E-NC'
      - warehouse Parent Group = 'WS Electro'
      - overlap NOT IN ('Zakka-Oral','Zakka-Appliances')
      - 集計：Ship(gross) + Move(gross/net)、キー：warehouse FN Code2（WS集計）
   ========================================================= */
d3_electro_nc_ship AS (
  SELECT
    sh.customer_org_mapping_key,
    ptc.category_key,
    ptc.category_name           AS category,
    cal.pg_quarter              AS quarter,

    wd.wh_fn_code2              AS fn_code,
    MAX(wd.wh_fn_name2)         AS fn_name,
    'WS'                        AS dc_ws,
    'Electro'                   AS z_e,

    SUM(sh.gross_transact_amt)  AS ship_giv,
    CAST(0.0 AS DOUBLE)         AS move_giv,
    CAST(0.0 AS DOUBLE)         AS move_niv
  FROM hive_metastore.japan_gold.shipments_fct_vw sh
  CROSS JOIN cal_bounds b
  JOIN cal_fy_half cal
    ON to_date(sh.day_date) = cal.day_date
  JOIN prod_to_cat ptc
    ON sh.prod_key = ptc.prod_key
  JOIN wh_dim wd
    ON CAST(sh.whkey AS BIGINT) = wd.whkey
  JOIN site_dim sd
    ON sh.ship_to_site_key = sd.site_key
  JOIN cust_dim cd
    ON sd.own_party_key = cd.cust_key
  WHERE sh.day_date >= b.start_ts AND sh.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.electro_flag2 = 'E-NC'
    AND wd.wh_parent_group = 'WS Electro'
    AND (ptc.overlap_76 IS NULL OR ptc.overlap_76 NOT IN ('Zakka-Oral','Zakka-Appliances'))
    AND wd.wh_fn_code2 IS NOT NULL
  GROUP BY
    sh.customer_org_mapping_key,
    ptc.category_key,
    ptc.category_name,
    cal.pg_quarter,
    wd.wh_fn_code2
),
d3_electro_nc_move AS (
  SELECT
    mv.customer_org_mapping_key,
    ptc.category_key,
    ptc.category_name            AS category,
    cal.pg_quarter               AS quarter,

    wd.wh_fn_code2               AS fn_code,
    MAX(wd.wh_fn_name2)          AS fn_name,
    'WS'                         AS dc_ws,
    'Electro'                    AS z_e,

    CAST(0.0 AS DOUBLE)          AS ship_giv,
    SUM(mv.gross_transact_amt)   AS move_giv,
    SUM(mv.net_transact_amt)     AS move_niv
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw mv
  CROSS JOIN cal_bounds b
  JOIN cal_fy_half cal
    ON to_date(mv.day_date) = cal.day_date
  JOIN prod_to_cat ptc
    ON mv.prod_key = ptc.prod_key
  JOIN wh_dim wd
    ON CAST(mv.whkey AS BIGINT) = wd.whkey
  JOIN site_dim sd
    ON mv.ship_to_site_key = sd.site_key
  JOIN cust_dim cd
    ON sd.own_party_key = cd.cust_key
  WHERE mv.day_date >= b.start_ts AND mv.day_date < b.end_ts_excl
    AND cd.cust_parent_group = 'RT'
    AND cd.electro_flag2 = 'E-NC'
    AND wd.wh_parent_group = 'WS Electro'
    AND (ptc.overlap_76 IS NULL OR ptc.overlap_76 NOT IN ('Zakka-Oral','Zakka-Appliances'))
    AND wd.wh_fn_code2 IS NOT NULL
  GROUP BY
    mv.customer_org_mapping_key,
    ptc.category_key,
    ptc.category_name,
    cal.pg_quarter,
    wd.wh_fn_code2
),

/* =========================================================
   9) D-1〜D-3 統合（Ship/MoveをUNION→最終集計）
   - FN×Category×Quarter×Z_E で必ず1行
   ========================================================= */
scenario_rows AS (
  SELECT * FROM d1_electro_dc_move
  UNION ALL SELECT * FROM d2_electro_dc_ship
  UNION ALL SELECT * FROM d3_electro_nc_ship
  UNION ALL SELECT * FROM d3_electro_nc_move
),

scenario_with_go AS (
  SELECT
    sr.fn_code,
    MAX(sr.fn_name) AS fn_name,   -- 要件：FN Nameはキーにしない
    sr.dc_ws,
    sr.category,
    sr.z_e,
    sr.quarter,
    sr.category_key,

    SUM(sr.ship_giv) AS ship_giv,
    SUM(sr.move_giv) AS move_giv,
    SUM(sr.move_niv) AS move_niv,

    -- ships_in_giv_customer_sales_gross
    SUM(sr.ship_giv + sr.move_giv) AS ships_in_giv,

    -- GARP/Outbound は mapping_key×category×quarter にぶら下がるので合算
    SUM(COALESCE(gq.garp, 0.0))     AS garp,
    SUM(COALESCE(oq.outbound, 0.0)) AS outbound
  FROM scenario_rows sr
  LEFT JOIN garp_qtr_cat gq
    ON sr.customer_org_mapping_key = gq.customer_org_mapping_key
   AND sr.category_key            = gq.category_key
   AND sr.quarter                 = gq.quarter
  LEFT JOIN outbound_qtr_cat oq
    ON sr.customer_org_mapping_key = oq.customer_org_mapping_key
   AND sr.category_key            = oq.category_key
   AND sr.quarter                 = oq.quarter
  GROUP BY
    sr.fn_code, sr.dc_ws, sr.category, sr.z_e, sr.quarter, sr.category_key
),

/* =========================================================
   10) FN×Category の Org_Code は end_date 最新を採用
   ========================================================= */
org_candidates AS (
  SELECT DISTINCT
    sr.fn_code,
    sr.category_key,
    gol.org5_key,
    gol.end_date
  FROM scenario_rows sr
  LEFT JOIN giv_org_latest_by_mapping gol
    ON sr.customer_org_mapping_key = gol.customer_org_mapping_key
   AND sr.category_key            = gol.category_key
),
org_fn_cat AS (
  SELECT
    t.fn_code,
    t.category_key,
    t.org5_key
  FROM (
    SELECT
      oc.fn_code,
      oc.category_key,
      oc.org5_key,
      ROW_NUMBER() OVER (
        PARTITION BY oc.fn_code, oc.category_key
        ORDER BY oc.end_date DESC NULLS LAST
      ) AS rn
    FROM org_candidates oc
    WHERE oc.org5_key IS NOT NULL
  ) t
  WHERE t.rn = 1
)

/* =========================================================
   Final Select（指定カラム名）
   ========================================================= */
SELECT
  s.fn_code                                AS fn_code,     -- (FN Code2)
  s.fn_name                                AS fn_name,     -- (FN Name2)
  s.dc_ws                                  AS dc_ws,       -- Customer:"DC" / Warehouse:"WS"
  s.category                               AS category,
  s.z_e                                    AS z_e,         -- Electro固定
  s.quarter                                AS quarter,     -- pg_quarter
  o.org2                                   AS org2,
  o.org3                                   AS org3,
  o.org4                                   AS org4,
  o.org5                                   AS org5,
  s.ships_in_giv                           AS ships_in_giv, -- ships_in_giv_customer_sales_gross
  s.ship_giv                               AS ship_giv,     -- shipment_giv_value_gross
  s.move_giv                               AS move_giv,     -- indirect_niv_value_gross
  s.move_niv                               AS move_niv,     -- indirect_niv_value_net
  s.garp                                   AS garp,         -- garp_value
  s.outbound                               AS outbound      -- giv_outbound
FROM scenario_with_go s
LEFT JOIN org_fn_cat ofc
  ON s.fn_code = ofc.fn_code
 AND s.category_key = ofc.category_key
LEFT JOIN org5_latest o
  ON ofc.org5_key = o.org5_key
ORDER BY
  s.fn_code, s.quarter, s.category, s.dc_ws;